Load the Silver table

In [0]:
df_silver = spark.table("retail_project.silver.sales_store_weather")

print("Rows:", df_silver.count())
df_silver.printSchema()

Add calendar-based features

In [0]:
from pyspark.sql.functions import dayofweek, month, year, weekofyear, col

df_features = (
    df_silver
    .withColumn("Year", year(col("Date")))
    .withColumn("Month", month(col("Date")))
    .withColumn("WeekOfYear", weekofyear(col("Date")))
)

display(df_features)

Add lag features (previous days' sales)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg

# Window: partition by store, ordered by date
store_window = Window.partitionBy("Store").orderBy("Date")

df_features = (
    df_features
    .withColumn("Sales_Lag1", lag("Sales", 1).over(store_window))
    .withColumn("Sales_Lag7", lag("Sales", 7).over(store_window))
)

display(df_features)

Add rolling average (7-day rolling mean per store)

In [0]:
rolling_window = Window.partitionBy("Store").orderBy("Date").rowsBetween(-7, -1)

df_features = df_features.withColumn(
    "Sales_RollingAvg7", avg("Sales").over(rolling_window)
)

display(df_features)

In [0]:
df_features.filter(col("Store") == 1).select(
    "Store", "Date", "Sales", "Sales_Lag1", "Sales_Lag7", "Sales_RollingAvg7"
).orderBy("Date").show(10)

Dropping nulls

In [0]:
df_features_clean = df_features.dropna(subset=["Sales_Lag1", "Sales_Lag7", "Sales_RollingAvg7"])

print("Before:", df_features.count())
print("After dropping early nulls:", df_features_clean.count())

Encode categorical columns

In [0]:
from pyspark.sql.functions import when, col

# StoreType and Assortment are single letters (a,b,c,d) - use StringIndexer for ML compatibility
from pyspark.ml.feature import StringIndexer

categorical_cols = ["StoreType", "Assortment", "StateHoliday"]

df_gold = df_features_clean
indexers = []

for c in categorical_cols:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_Index", handleInvalid="keep")
    df_gold = indexer.fit(df_gold).transform(df_gold)

display(df_gold)

Drop columns not needed for modeling

In [0]:
df_gold_final = df_gold.drop("StoreType", "Assortment", "StateHoliday", "PromoInterval", "Customers")

display(df_gold_final)
df_gold_final.printSchema()

Create schema if it doesnt exist

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.gold")

Write the Gold table

In [0]:
df_gold_final.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.gold.sales_features"
)

print("Gold table written: retail_project.gold.sales_features")

Test schema

In [0]:
df_gold_final.printSchema()
print("Final row count:", df_gold_final.count())